# Advanced Analysis: RFM, Basket, Cohort & Forecast

**Project:** E-commerce Sales Data Analysis & Machine Learning  
**Author:** Muhammad Iqbal Fadel  
**Source:** `scripts/02_advanced_analysis.py`

In [ ]:
%matplotlib inline

from __future__ import annotations

import os
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

In [ ]:
ROOT = os.path.abspath('..')
DATA_PATH = os.path.join(ROOT, 'data', 'processed', 'Sales_Transaction_v4a_cleaned.csv')
DOC_DIR = os.path.join(ROOT, 'outputs', 'figures', 'eda')
CSV_DIR = os.path.join(ROOT, 'outputs', 'data', 'analysis')
ADV_DIR = os.path.join(ROOT, 'outputs', 'data', 'analysis')
for path in (DOC_DIR, CSV_DIR, ADV_DIR):
    os.makedirs(path, exist_ok=True)

In [ ]:
def savefig(path: str) -> None:
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()

In [ ]:
def load_data() -> pd.DataFrame:
    df = pd.read_csv(DATA_PATH, low_memory=False)
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    if 'YearMonth' not in df.columns:
        df['YearMonth'] = df['Date'].dt.to_period('M').astype(str)
    if 'TotalAmount' not in df.columns:
        df['TotalAmount'] = df['Price'] * df['Quantity']
    return df

In [ ]:
def customer_segmentation(df: pd.DataFrame) -> None:
    rfm_csv = os.path.join(CSV_DIR, 'rfm_summary.csv')
    if not os.path.exists(rfm_csv):
        return

    rfm = pd.read_csv(rfm_csv)
    strategies = pd.DataFrame(
        [
            ('Champions', 'Pertahankan dengan loyalty program, early access, dan upsell produk premium', 'Tinggi'),
            ('Loyal', 'Berikan bundling, rekomendasi produk, dan promo repeat order', 'Tinggi'),
            ('At Risk', 'Kirim campaign win-back, diskon terbatas, dan reminder produk favorit', 'Sedang'),
            ('Hibernating', 'Gunakan reactivation campaign dengan penawaran sederhana', 'Sedang'),
            ('Potential', 'Dorong pembelian kedua dengan voucher dan rekomendasi produk terkait', 'Sedang'),
        ],
        columns=['Segment', 'RecommendedAction', 'Priority']
    )
    strategy_path = os.path.join(CSV_DIR, 'rfm_strategy_recommendations.csv')
    strategies.to_csv(strategy_path, index=False)

    merged = rfm.merge(strategies, on='Segment', how='left')
    print('\nCustomer segmentation summary:')
    print(merged[['Segment', 'Customers', 'Avg_Recency', 'Avg_Frequency', 'Avg_Monetary', 'RecommendedAction']])

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=rfm, x='Segment', y='Customers', palette='mako', ax=ax)
    ax.set_title('Customer Segments by Count')
    ax.tick_params(axis='x', rotation=25)
    savefig(os.path.join(DOC_DIR, 'rfm_segment_counts.png'))

    fig, ax = plt.subplots(figsize=(9, 4))
    sns.heatmap(rfm.set_index('Segment')[['Avg_Recency', 'Avg_Frequency', 'Avg_Monetary']], annot=True, fmt='.1f', cmap='YlGnBu', ax=ax)
    ax.set_title('RFM Segment Averages')
    savefig(os.path.join(DOC_DIR, 'rfm_segment_averages_heatmap.png'))

In [ ]:
def product_analysis(df: pd.DataFrame) -> None:
    revenue = df.groupby('ProductName', as_index=False)['TotalAmount'].sum().sort_values('TotalAmount', ascending=False)
    quantity = df.groupby('ProductName', as_index=False)['Quantity'].sum().sort_values('Quantity', ascending=False)
    revenue.to_csv(os.path.join(CSV_DIR, 'product_revenue_summary.csv'), index=False)
    quantity.to_csv(os.path.join(CSV_DIR, 'product_quantity_summary.csv'), index=False)

    top_rev = revenue.head(20)
    top_qty = quantity.head(20)

    fig, ax = plt.subplots(figsize=(12, 8))
    sns.barplot(data=top_rev, y='ProductName', x='TotalAmount', palette='viridis', ax=ax)
    ax.set_title('Top 20 Products by Revenue')
    ax.set_xlabel('Revenue (GBP)')
    savefig(os.path.join(DOC_DIR, 'top20_products_by_revenue.png'))

    fig, ax = plt.subplots(figsize=(12, 8))
    sns.barplot(data=top_qty, y='ProductName', x='Quantity', palette='crest', ax=ax)
    ax.set_title('Top 20 Products by Quantity')
    ax.set_xlabel('Quantity Sold')
    savefig(os.path.join(DOC_DIR, 'top20_products_by_quantity.png'))

In [ ]:
def basket_analysis(df: pd.DataFrame) -> None:
    top_products = df['ProductName'].value_counts().head(100).index
    basket_df = df[df['ProductName'].isin(top_products)].copy()
    transactions = (
        basket_df.groupby('TransactionNo')['ProductName']
        .apply(lambda s: sorted(set(s.dropna().tolist())))
        .tolist()
    )

    pair_counts: dict[tuple[str, str], int] = {}
    for items in transactions:
        if len(items) < 2:
            continue
        for pair in combinations(items, 2):
            pair_counts[pair] = pair_counts.get(pair, 0) + 1

    pair_summary = pd.DataFrame(
        [(a, b, count) for (a, b), count in pair_counts.items()],
        columns=['ProductA', 'ProductB', 'CoOccurrenceCount']
    ).sort_values('CoOccurrenceCount', ascending=False)

    pair_path = os.path.join(CSV_DIR, 'basket_top_pairs.csv')
    pair_summary.to_csv(pair_path, index=False)

    if not pair_summary.empty:
        top_pairs = pair_summary.head(20).copy()
        top_pairs['Pair'] = top_pairs['ProductA'] + ' + ' + top_pairs['ProductB']

        fig, ax = plt.subplots(figsize=(12, 8))
        sns.barplot(data=top_pairs, y='Pair', x='CoOccurrenceCount', palette='rocket', ax=ax)
        ax.set_title('Top Product Pairs in Same Transaction')
        ax.set_xlabel('Co-occurrence Count')
        savefig(os.path.join(DOC_DIR, 'basket_top_pairs.png'))

In [ ]:
def cohort_analysis(df: pd.DataFrame) -> None:
    cohort_source = df.dropna(subset=['CustomerNo', 'Date']).copy()
    cohort_source['OrderMonth'] = cohort_source['Date'].dt.to_period('M')
    cohort_source['CohortMonth'] = cohort_source.groupby('CustomerNo')['Date'].transform('min').dt.to_period('M')
    cohort_source['CohortIndex'] = (
        (cohort_source['OrderMonth'].dt.year - cohort_source['CohortMonth'].dt.year) * 12
        + (cohort_source['OrderMonth'].dt.month - cohort_source['CohortMonth'].dt.month)
        + 1
    )

    cohort_counts = cohort_source.groupby(['CohortMonth', 'CohortIndex'])['CustomerNo'].nunique().reset_index()
    cohort_pivot = cohort_counts.pivot(index='CohortMonth', columns='CohortIndex', values='CustomerNo').fillna(0)
    retention = cohort_pivot.divide(cohort_pivot.iloc[:, 0], axis=0) * 100

    cohort_counts.to_csv(os.path.join(CSV_DIR, 'cohort_counts.csv'), index=False)
    retention.round(2).to_csv(os.path.join(CSV_DIR, 'cohort_retention.csv'))

    fig, ax = plt.subplots(figsize=(12, 6))
    sns.heatmap(retention, annot=True, fmt='.0f', cmap='Blues', ax=ax)
    ax.set_title('Customer Retention Cohort Heatmap (%)')
    ax.set_xlabel('Cohort Month Number')
    ax.set_ylabel('Cohort Month')
    savefig(os.path.join(DOC_DIR, 'cohort_retention_heatmap.png'))

In [ ]:
def forecasting(df: pd.DataFrame) -> None:
    monthly = df.groupby('YearMonth', as_index=False)['TotalAmount'].sum().sort_values('YearMonth')
    monthly['MonthIndex'] = np.arange(len(monthly))
    x = monthly['MonthIndex'].to_numpy()
    y = monthly['TotalAmount'].to_numpy()

    if len(monthly) < 2:
        return

    slope, intercept = np.polyfit(x, y, 1)
    future_steps = np.arange(len(monthly), len(monthly) + 3)
    future_values = slope * future_steps + intercept

    forecast_df = pd.DataFrame(
        {
            'YearMonth': list(monthly['YearMonth']) + [f'Forecast_{i + 1}' for i in range(len(future_steps))],
            'TotalAmount': list(monthly['TotalAmount']) + list(future_values),
            'Type': ['Actual'] * len(monthly) + ['Forecast'] * len(future_steps)
        }
    )
    forecast_df.to_csv(os.path.join(CSV_DIR, 'monthly_forecast.csv'), index=False)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(monthly['MonthIndex'], monthly['TotalAmount'], marker='o', label='Actual')
    ax.plot(future_steps, future_values, marker='o', linestyle='--', label='Forecast')
    ax.set_title('Simple Monthly Sales Forecast (Linear Trend)')
    ax.set_xlabel('Month Index')
    ax.set_ylabel('Total Amount (GBP)')
    ax.legend()
    savefig(os.path.join(DOC_DIR, 'monthly_sales_forecast.png'))


df = load_data()
print('Loaded data:', df.shape)
customer_segmentation(df)
product_analysis(df)
basket_analysis(df)
cohort_analysis(df)
forecasting(df)
print('Advanced analyses saved to Dokumentasi and Dokumentasi/csv')